# golo — comparar Python vs Dart (CryptoVault V2)
Este cuaderno **clona el repo**, ejecuta las pruebas Python y comprueba igualdad con una `pass` que te pide.
El Dart (`vault.dart`) usa el mismo envelope `PRBX`, así que un archivo cifrado en Python se descifra en Dart y viceversa.

In [ ]:
import os, sys
REPO_URL = "https://github.com/elmasber-ma/golo.git"
DEST = "/tmp/golo"
if not os.path.isdir(DEST):
    !git clone $REPO_URL $DEST
else:
    print("ya clonado", DEST)
sys.path.insert(0, DEST)
%cd $DEST
!ls -la

In [ ]:
!pip -q install cryptography 2>&1 | tail -n 2
!python test_vault.py

In [ ]:
import getpass
from toolsec import Vault, is_envelope

pw = getpass.getpass("pass: ")
v = Vault(pw)
msg = b"mensaje de prueba para igualdad"
enc = v.encrypt(msg)
print("envelope:", enc[:4], "ver:", enc[4], "len:", len(enc))
print("salt:", enc[5:21].hex())
print("nonce:", enc[21:33].hex())
print("tag:", enc[-16:].hex())
dec = v.decrypt(enc)
print("igualdad:", dec == msg)
assert dec == msg, "FALLO igualdad"
# pass incorrecta debe fallar
mal = Vault("pass-equivocada").decrypt(enc)
print("pass incorrecta ->", mal)
assert mal is None
print("OK igualdad comprobada")

## Prueba Dart en Colab (cruce Python <-> Dart)
Instala Dart SDK, cifra con Dart y descifra con Python y viceversa, todo con la misma `pass`.

In [ ]:
!which dart || (sudo apt-get update -qq && sudo apt-get install -y -qq apt-transport-https wget gnupg) 2>&1 | tail -n 1
!which dart || (wget -qO- https://dl-ssl.google.com/linux/linux_signing_key.pub | sudo gpg --dearmor -o /usr/share/keyrings/dart.gpg && echo 'deb [signed-by=/usr/share/keyrings/dart.gpg arch=amd64] https://storage.googleapis.com/download.dartlang.org/linux/debian stable main' | sudo tee /etc/apt/sources.list.d/dart_stable.list && sudo apt-get update -qq && sudo apt-get install -y -qq dart) 2>&1 | tail -n 1
!dart --version
!dart pub get 2>&1 | tail -n 2

In [ ]:
import os
os.environ["GOLO_PASS"] = pw  # reutiliza la pass pedida arriba
!printf 'mensaje de prueba para igualdad' > msg.bin
!dart run vault.dart enc "$GOLO_PASS" msg.bin msg_dart.prbx
!dart run vault.dart dec "$GOLO_PASS" msg_dart.prbx msg_dart.out && cmp msg.bin msg_dart.out && echo "DART IGUALDAD OK"
!ls -la msg*.prbx msg*.out

In [ ]:
# Cruce: Dart cifra -> Python descifra, y Python cifra -> Dart descifra
from toolsec import Vault
v = Vault(pw)
dart_blob = open("msg_dart.prbx", "rb").read()
print("dart->python igualdad:", v.decrypt(dart_blob) == b"mensaje de prueba para igualdad")
assert v.decrypt(dart_blob) == b"mensaje de prueba para igualdad"
open("msg_py.bin", "wb").write(b"mensaje de prueba para igualdad")
!python cli.py enc "$GOLO_PASS" msg_py.bin msg_py.prbx && dart run vault.dart dec "$GOLO_PASS" msg_py.prbx msg_py.out && cmp msg_py.bin msg_py.out && echo "CRUCE PY<->DART OK"